# Glashelder: CLIP-based search interface

This notebook walks you through the setup and usage of a search inteface for the Glashelder images dataset.

The idea is as follows: create embeddings of all images in a latent space (a sort of semantically meaningful vectorspace) with CLIP ViT.

The CLIP model also has the ability to encode text into the same latent space.

When you supply some text (a prompt) (eg. "image of a cat on a chair", "people laughing", "starfish") the embedding of the text is compared to all embeddings of the images and the best matches are shown.

## Configurations

Import all necessary python libraries by running the cell below:

In [1]:
import glob
from pathlib import Path

from cluster.cli import (
    find_similar,
    perform_search,
)
from common import PROJECT_ROOT
from embed import get_device, get_embedder

The variabels below specify the key components necessary to run the CLIP model:
- `MODEL`: the CLIP model to get from huggingface
- `PACKAGE_ROOT`: filesystem path of the directory with files related to the `cluster` experiment
- `EMBEDDINGS_PATH`: the location where the embeddings will be saved to in a file
- `IMAGE_DIR`: the path where you store your Glashelder images. **Modify the path below to where your Glashelder images are located**.

In [5]:
MODEL = "laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K"
PACKAGE_ROOT = PROJECT_ROOT / "cluster"
EMBEDDINGS_PATH = PACKAGE_ROOT / "embeddings.pkl"
IMAGE_DIR = "/mnt/share/shares/glashelder/assets"  # MODIFY ME

## Setup model and preprocessor

Download the model and set it up for your device (cuda, cpu, mps):

In [ ]:
device = get_device()
embedder = get_embedder("clip")
embedder.load(device)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/workspace/.venv/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/workspace/.venv/lib/python3.13/site-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/workspace/.venv/lib/python3.13/site-packages/transformers/safeten

## Make image embeddings

A random selection of images is chosen from within the image directory (defined above in `ÌMAGE_DIR`). The amount of images is limited here to a dataset of 4000 for speed. If you want a larger dataset to look through, increase the variable, `count`.

In [7]:
# this makes a 'random' selection of images in the dataset of limited size, for speed
count = 4000 #If you want a larger dataset to look through, increase this variable
image_paths = sorted(glob.glob(f"{IMAGE_DIR}/*.jpg"))[:count]

Generate the embeddings of the images. Depending on the size of your dataset with pictures this can take some time.

In [8]:
embeddings, image_paths = embedder.encode_images(image_paths, batch_size=64)
embeddings.shape  # count vectors of length 768

torch.Size([4000, 768])

## Define web server for search interface

The following code starts a web server on your computer with a search interface.

When the code is running, you can see the webpage at [localhost:5000](http://localhost:5000)

In [9]:
from flask import Flask, render_template, request, send_from_directory

app = Flask(
    __name__,
    template_folder=PACKAGE_ROOT / "templates",
    static_folder=PACKAGE_ROOT / "static",
)

image_embeddings = embeddings.to(device)


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/search", methods=["POST"])
def search_endpoint():
    prompt = request.form.get("prompt", "").strip()
    top_k = int(request.form.get("top_k", 12))

    if not prompt:
        return render_template(
            "results.html",
            error="Please enter a search prompt",
            results=[],
            query_type="text",
            query_value="",
        )

    try:
        text_embedding = embedder.encode_text(prompt)
        if text_embedding is None:
            return render_template(
                "results.html",
                error="Model does not support text encoding",
                results=[],
                query_type="text",
                query_value=prompt,
            )
        text_embedding = text_embedding.to(device)
        top_indices, top_values = perform_search(
            top_k, text_embedding, image_embeddings, image_paths
        )

        results = []
        for rank, (idx, score) in enumerate(
            zip(top_indices.cpu().numpy(), top_values.cpu().numpy(), strict=False), 1
        ):
            results.append(
                {
                    "rank": rank,
                    "index": int(idx),
                    "score": float(score),
                    "filename": Path(image_paths[idx]).name,
                }
            )

        return render_template(
            "results.html",
            results=results,
            error=None,
            query_type="text",
            query_value=prompt,
        )

    except Exception as e:
        return render_template(
            "results.html",
            error=f"Search failed: {e!s}",
            results=[],
            query_type="text",
            query_value="",
        )


@app.route("/similar/<int:index>")
def similar_fullpage(index):
    """Find images similar to the image at the given index"""
    top_k = int(request.args.get("top_k", 12))
    if index < 0 or index >= len(image_paths):
        return render_template(
            "results.html",
            error="Image not found",
            results=[],
            query_type="similar",
            query_value=index,
        )
    try:
        top_indices, top_values = find_similar(
            index, top_k, image_embeddings, image_paths
        )
        results = []
        for rank, (idx, score) in enumerate(
            zip(top_indices.cpu().numpy(), top_values.cpu().numpy(), strict=False), 1
        ):
            results.append(
                {
                    "rank": rank,
                    "index": int(idx),
                    "score": float(score),
                    "filename": Path(image_paths[idx]).name,
                }
            )

        query_filename = Path(image_paths[index]).name
        return render_template(
            "similar.html",
            results=results,
            error=None,
            query_type="similar",
            query_value=index,
            query_filename=query_filename,
        )

    except Exception as e:
        return render_template(
            "similar.html",
            error=f"Search failed: {e!s}",
            results=[],
            query_type="similar",
            query_value=index,
        )


@app.route("/api/similar/<int:index>")
def similar_endpoint(index):
    """Find images similar to the image at the given index"""
    top_k = int(request.args.get("top_k", 12))

    if index < 0 or index >= len(image_paths):
        return render_template(
            "results.html",
            error="Image not found",
            results=[],
            query_type="similar",
            query_value=index,
        )

    try:
        top_indices, top_values = find_similar(
            index, top_k, image_embeddings, image_paths
        )

        results = []
        for rank, (idx, score) in enumerate(
            zip(top_indices.cpu().numpy(), top_values.cpu().numpy(), strict=False), 1
        ):
            results.append(
                {
                    "rank": rank,
                    "index": int(idx),
                    "score": float(score),
                    "filename": Path(image_paths[idx]).name,
                }
            )

        query_filename = Path(image_paths[index]).name
        return render_template(
            "results.html",
            results=results,
            error=None,
            query_type="similar",
            query_value=index,
            query_filename=query_filename,
        )

    except Exception as e:
        return render_template(
            "results.html",
            error=f"Search failed: {e!s}",
            results=[],
            query_type="similar",
            query_value=index,
        )


@app.route("/image/<int:index>")
def serve_image(index):
    """Serve image by index"""
    if index < 0 or index >= len(image_paths):
        return "Image not found", 404

    image_path = Path(image_paths[index])
    return send_from_directory(image_path.parent, image_path.name)


app.run(
    host="0.0.0.0", port=5000, debug=True, use_reloader=False
)  # use_reloader=False is important!

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.201.5.2:5000
Press CTRL+C to quit
